# DisasterAI — Egitilmis Modeli Test Et

**Kaggle'da calistir.** Sag panelden IKI girdi ekle:
1. **Add Input > Your Datasets > disasterai-corpora** (test.jsonl icin)
2. **Add Input > Your Work / Notebook Output > qlora_egitim** (adaptor icin)

Accelerator: **GPU T4 x2** (kod tek karta sabitler).

In [ ]:
#@title 1) Kurulum (~3 dk)
!pip -q install -U "transformers>=4.44" "peft>=0.13" "bitsandbytes>=0.44" accelerate
print("kurulum tamam")

In [ ]:
#@title 2) Test verisini ve adaptoru bul
import os, glob, json

test_yol = None
adaylar = []
for kok, klasorler, dosyalar in os.walk("/kaggle/input"):
    if "test.jsonl" in dosyalar and test_yol is None:
        test_yol = os.path.join(kok, "test.jsonl")
    if "adapter_config.json" in dosyalar:
        adaylar.append(kok)

# Once nihai kayit (disasterai_lora), yoksa en son checkpoint
nihai = [a for a in adaylar if "disasterai_lora" in a]
if nihai:
    adaptor_yol = nihai[0]
elif adaylar:
    adaptor_yol = sorted(adaylar,
        key=lambda p: int(p.split("-")[-1]) if p.split("-")[-1].isdigit() else 0)[-1]
else:
    adaptor_yol = None

print("test.jsonl :", test_yol)
print("adaptor    :", adaptor_yol)
print("adaptor icerigi:", sorted(os.listdir(adaptor_yol))[:8] if adaptor_yol else "-")
if not test_yol or not adaptor_yol:
    raise SystemExit("Eksik girdi var — iki Input'u da ekle.")

ornekler = [json.loads(s) for s in open(test_yol, encoding="utf-8")]
print("test ornegi sayisi:", len(ornekler))

In [ ]:
#@title 3) Modeli yukle: taban 4-bit + bizim adaptor (~10 dk)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "ytu-ce-cosmos/Turkish-Llama-8b-Instruct-v0.1"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(adaptor_yol)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

taban = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map={"": 0})
model = PeftModel.from_pretrained(taban, adaptor_yol)
model.eval()
print("model + adaptor hazir")

In [ ]:
#@title 4) Metin uretim fonksiyonu (dogru durdurma jetonu ile)
import torch

# Llama-3 sohbet sablonu turu <|eot_id|> ile bitirir; sadece <|end_of_text|>
# verilirse model durmayip ikinci bir yanit uydurmaya baslar.
dur_jetonlari = [tokenizer.eos_token_id]
for ad in ["<|eot_id|>", "<|end_of_text|>"]:
    jid = tokenizer.convert_tokens_to_ids(ad)
    if isinstance(jid, int) and jid >= 0 and jid not in dur_jetonlari:
        dur_jetonlari.append(jid)
print("durdurma jetonlari:", dur_jetonlari)

def uret(prompt, azami_token=400, sicaklik=0.7):
    mesajlar = [{"role": "user", "content": prompt}]
    metin = tokenizer.apply_chat_template(mesajlar, tokenize=False,
                                          add_generation_prompt=True)
    girdi = tokenizer(metin, return_tensors="pt").to(model.device)
    with torch.no_grad():
        cikti = model.generate(**girdi, max_new_tokens=azami_token,
                               temperature=sicaklik, do_sample=True,
                               eos_token_id=dur_jetonlari,
                               pad_token_id=tokenizer.eos_token_id)
    yeni_jetonlar = cikti[0][girdi["input_ids"].shape[1]:]
    return tokenizer.decode(yeni_jetonlar, skip_special_tokens=True).strip()

print("hazir")

In [ ]:
#@title 5) 4 ornek uret ve OKU (her tondan bir tane)
gorulen = set()
secilenler = []
for o in ornekler:
    if o["ton"] not in gorulen:
        gorulen.add(o["ton"]); secilenler.append(o)
    if len(secilenler) == 4:
        break

uretilenler = []
for o in secilenler:
    metin = uret(o["prompt"])
    uretilenler.append((o, metin))
    print("=" * 78)
    print(f"[{o['ton'].upper()}] {o['afet_tipi']} — {o['konum_id']}")
    print("-" * 78)
    print("BIZIM MODEL:")
    print(metin)
    print("-" * 78)
    print("OGRETMEN (referans):")
    print(o["cikti"][:600])
    print()

In [ ]:
#@title 6) SAYI SADAKATI olcumu (test kumesinin ilk 40 ornegi, ~15 dk)
import re

SAYI = re.compile(r"(?<![\w.,])(\d+(?:[.,]\d+)?)(?![\w])")
BINLIK = re.compile("(?<=\\d)[ \u00a0\u2009\u202f](?=\\d{3}(?!\\d))")

def sayilar(metin):
    temiz = BINLIK.sub("", metin)
    out = set()
    for m in SAYI.finditer(temiz):
        try:
            out.add(round(float(m.group(1).replace(",", ".")), 2))
        except ValueError:
            pass
    return out

def izinli_kume(prompt):
    izin = set()
    for s in sayilar(prompt):
        izin.update({s, round(s, 1), float(round(s))})
        if abs(s) >= 100:
            izin.update({round(s / 1000, 2), round(s / 1000, 1),
                         float(round(s / 10) * 10), float(round(s / 100) * 100)})
    izin.update({0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 2.9, 3.9, 25.0, 30.0})
    return izin

N = 40
uydurma_iceren = 0
toplam_sayi = 0
toplam_uydurma = 0
kayitlar = []

for i, o in enumerate(ornekler[:N]):
    metin = uret(o["prompt"])
    izin = izinli_kume(o["prompt"])
    bulunan = sayilar(metin)
    uydurma = [s for s in bulunan
               if not any(abs(s - iz) <= max(0.05, min(abs(iz) * 0.01, 10.0)) for iz in izin)]
    toplam_sayi += len(bulunan)
    toplam_uydurma += len(uydurma)
    if uydurma:
        uydurma_iceren += 1
    kayitlar.append({"konum_id": o["konum_id"], "ton": o["ton"],
                     "uydurma": uydurma, "cikti": metin})
    if (i + 1) % 10 == 0:
        print(f"  ... {i+1}/{N}")

print()
print("=" * 60)
print(f"Uydurma sayi ICEREN yanit orani : %{100*uydurma_iceren/N:.1f}  ({uydurma_iceren}/{N})")
print(f"Uydurma sayi / toplam sayi      : %{100*toplam_uydurma/max(toplam_sayi,1):.1f}  ({toplam_uydurma}/{toplam_sayi})")
print("=" * 60)

with open("/kaggle/working/test_sonuclari.json", "w", encoding="utf-8") as f:
    json.dump(kayitlar, f, ensure_ascii=False, indent=1)
print("ayrinti kaydedildi: /kaggle/working/test_sonuclari.json")